In [ ]:
# ============================================================
# CELL 1 — Install dependencies
# ============================================================
!pip install openai --quiet

print("✅ Dependencies installed")

✅ Dependencies installed


In [ ]:
# ============================================================
# CELL 2 — Load secrets
# ============================================================
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

print("✅ Secrets loaded")

✅ Secrets loaded


In [ ]:
# ============================================================
# CELL 3 — Node 1: intake
#
# This is the ONLY thing this cell does — decide if the
# requirement is clear enough to design from. Nothing else
# exists yet. No state, no graph, no other node.
# ============================================================

from openai import OpenAI
import json

CLARITY_PROMPT = """You are a senior Salesforce Data Architect reviewing a
project requirement before beginning design work.

Decide whether the following contains enough information to design from.

Requirement: {requirement}
Process Flow: {process_flow}
Edge Cases: {edge_cases}
Current Process: {current_process}

A requirement is CLEAR ENOUGH if the business problem, the basic flow of
events, and any systems it touches are reasonably understandable.

A requirement is TOO VAGUE if the problem could be interpreted multiple
ways, or the flow is missing entirely, or it's just a one-line wish with
no detail.

Respond in this exact JSON format only, no other text:
{{
  "is_clear": true or false,
  "clarifying_questions": ["question 1", "question 2"]
}}

If is_clear is true, clarifying_questions must be an empty list."""


def intake(requirement: str, process_flow: str, edge_cases: str
           , current_process: str) -> dict:
    """
    Takes the five raw input strings.
    Returns {"is_clear": bool, "clarifying_questions": [...]}.
    """
    client = OpenAI()

    prompt = CLARITY_PROMPT.format(
        requirement=requirement,
        process_flow=process_flow,
        edge_cases=edge_cases,
        current_process=current_process,
    )

    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=300,
        temperature=0.1
    )

    raw = resp.choices[0].message.content.strip()
    return json.loads(raw)


print("✅ intake() defined")

✅ intake() defined


In [ ]:
# ============================================================
# CELL 4 — Test intake()
# ============================================================

result_clear = intake(
    requirement="Track loan applications from submission through underwriting approval.",
    process_flow="Applicant submits, underwriter reviews, credit check runs, decision made, applicant notified.",
    edge_cases="Some applications need co-applicants.",
    current_process="Currently tracked in spreadsheets."
)
print(result_clear)

result_vague = intake(
    requirement="We need something to track our stuff better.",
    process_flow="People do things and it gets tracked.",
    edge_cases="Not sure.",
    current_process="Spreadsheets."
)
print(result_vague)

{'is_clear': True, 'clarifying_questions': []}
{'is_clear': False, 'clarifying_questions': ['What specific items or data do you need to track?', 'What are the key events or actions that need to be captured in the tracking process?', 'What specific information is currently being tracked in the spreadsheets?', 'Who are the users involved in this process, and what are their roles?', "What are the desired outcomes or metrics for tracking 'stuff' better?"]}


In [ ]:
# ============================================================
# CELL 5 — Tool: validate_object_name
#
# A standalone function. Not a node. Not wired to anything yet.
# Checks one Salesforce naming rule: custom objects must end
# in __c, can't be a reserved name, can't start with a digit.
# ============================================================

RESERVED_OBJECT_NAMES = [
    "User", "Profile", "PermissionSet", "Group", "Territory",
    "Forecast", "Contract", "Order", "Asset", "Case", "Account",
    "Contact"
]


def validate_object_name(name: str) -> dict:
    errors = []

    if not name.endswith("__c"):
        errors.append("Custom object names must end with __c")

    base = name.replace("__c", "")
    if base and base[0].isdigit():
        errors.append("Object name cannot start with a number")

    if base in RESERVED_OBJECT_NAMES:
        errors.append(f"{base} is a reserved Salesforce object name")

    return {"name": name, "valid": len(errors) == 0, "errors": errors}


print("✅ validate_object_name() defined")

✅ validate_object_name() defined


In [ ]:
# ============================================================
# CELL 6 — Test validate_object_name() directly
# ============================================================

print(validate_object_name("Loan_Application__c"))   # expect valid
print(validate_object_name("Loan_Application"))       # expect invalid — no __c
print(validate_object_name("User__c"))                 # expect invalid — reserved

{'name': 'Loan_Application__c', 'valid': True, 'errors': []}
{'name': 'Loan_Application', 'valid': False, 'errors': ['Custom object names must end with __c']}
{'name': 'User__c', 'valid': False, 'errors': ['User is a reserved Salesforce object name']}


In [ ]:
# ============================================================
# CELL 7 — Tool: validate_field_type
#
# Checks a field type string against Salesforce's actual
# valid field type names.
# ============================================================

VALID_FIELD_TYPES = [
    "Text", "Number", "Currency", "Date", "DateTime",
    "Picklist", "MultiSelectPicklist", "Checkbox", "Email",
    "Phone", "URL", "TextArea", "LongTextArea", "Percent"
]


def validate_field_type(field_type: str) -> dict:
    relationship_words = ["lookup to", "master-detail to", "master detail to"]

    if any(w in field_type.lower() for w in relationship_words):
        return {
            "field_type": field_type,
            "valid": False,
            "error": f"'{field_type}' is a relationship, it belongs in "
                     f"relationships, not fields."
        }

    valid = field_type in VALID_FIELD_TYPES
    return {
        "field_type": field_type,
        "valid": valid,
        "error": None if valid else f"'{field_type}' is not a valid Salesforce field type"
    }


print("✅ validate_field_type() defined")

✅ validate_field_type() defined


In [ ]:
# ============================================================
# CELL 8 — Test validate_field_type() directly
# ============================================================

print(validate_field_type("Currency"))                     # expect valid
print(validate_field_type("Long Text Area"))                # expect invalid — wrong name
print(validate_field_type("Lookup to Account"))              # expect invalid — relationship, not a type

{'field_type': 'Currency', 'valid': True, 'error': None}
{'field_type': 'Long Text Area', 'valid': False, 'error': "'Long Text Area' is not a valid Salesforce field type"}
{'field_type': 'Lookup to Account', 'valid': False, 'error': "'Lookup to Account' is a relationship, it belongs in relationships, not fields."}


In [ ]:
# ============================================================
# CELL 9 — Node 2: design (updated)
#
# Same as before, plus a standing instruction to prefer
# existing standard Salesforce objects over inventing new
# custom ones when a standard object already fits.
# ============================================================

import re

DESIGN_PROMPT = """You are a senior Salesforce Data Architect.

Your job is ONLY to design the data model — objects, fields, relationships.
You do NOT design automation, triggers, Flows, or Apex. That is someone
else's job. Do not mention or suggest automation at all.

Design philosophy, always apply:
- Relationships should be composable: flexible and reusable, not rigid
- Field and object structure should protect data reliability and integrity
- ALWAYS prefer existing standard Salesforce objects over inventing new
  custom objects when a standard object already fits the need. Standard
  objects available to you:
    - Account: any organization, company, or business entity
    - Contact: any individual person associated with an Account
    - User: any internal Salesforce user, such as an employee or staff member
  Only create a custom object when the data genuinely does not fit any
  standard object above.

REQUIREMENT: {requirement}
PROCESS FLOW: {process_flow}
EDGE CASES: {edge_cases}
CURRENT PROCESS: {current_process}
{override_section}

Rules:
- Custom object API names must end with __c
- Fields array entries must be in the format "Field_Name__c (Type)"
- Valid field types ONLY: Text, Number, Currency, Date, DateTime, Picklist,
  MultiSelectPicklist, Checkbox, Email, Phone, URL, TextArea, LongTextArea, Percent
- Relationships (Lookup to X, Master-Detail to Y) go ONLY in the
  relationships array, never in fields
- Prefer Lookup over Master-Detail unless the child truly cannot
  exist without the parent

Respond in this exact JSON format only, no other text:
{{
  "data_model": [
    {{
      "object_name": "Human readable name",
      "api_name": "API_Name__c",
      "fields": ["Field_Name__c (Type)"],
      "relationships": ["Lookup to Account"]
    }}
  ],
  "alternative_considered": "only fill this in if a real trade-off was made, otherwise empty string"
}}"""


def design(requirement: str, process_flow: str, edge_cases: str,
           current_process: str, override_note: str = "") -> dict:
    client = OpenAI()

    override_section = f"OVERRIDE NOTE: {override_note}" if override_note else ""

    prompt = DESIGN_PROMPT.format(
        requirement=requirement,
        process_flow=process_flow,
        edge_cases=edge_cases,
        current_process=current_process,
        override_section=override_section
    )

    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1200,
        temperature=0.3
    )

    raw = resp.choices[0].message.content.strip()
    raw = re.sub(r"^```[a-z]*\n?|\n?```$", "", raw)
    parsed = json.loads(raw)

    for obj in parsed.get("data_model", []):
        name_check = validate_object_name(obj["api_name"])
        obj["naming_valid"]  = name_check["valid"]
        obj["naming_errors"] = name_check["errors"]

        field_results = []
        for field_str in obj.get("fields", []):
            if "(" in field_str and ")" in field_str:
                field_type = field_str.split("(")[1].split(")")[0].strip()
                field_results.append(validate_field_type(field_type))
        obj["field_check_results"] = field_results

    return parsed


print("✅ design() defined")

✅ design() defined


In [ ]:
# ============================================================
# CELL 10 — Test design() directly, on the same clear
# requirement intake() already confirmed was clear
# ============================================================

result = design(
    requirement="Track loan applications from submission through underwriting approval.",
    process_flow="Applicant submits, underwriter reviews, credit check runs, decision made, applicant notified.",
    edge_cases="Some applications need co-applicants.",
    current_process="Currently tracked in spreadsheets."
)

print(json.dumps(result, indent=2))

{
  "data_model": [
    {
      "object_name": "Loan Application",
      "api_name": "Loan_Application__c",
      "fields": [
        "Application_Date__c (Date)",
        "Loan_Amount__c (Currency)",
        "Status__c (Picklist)",
        "Decision_Date__c (Date)",
        "Notes__c (LongTextArea)"
      ],
      "relationships": [
        "Lookup to Account",
        "Lookup to Contact",
        "Lookup to User"
      ],
      "naming_valid": true,
      "naming_errors": [],
      "field_check_results": [
        {
          "field_type": "Date",
          "valid": true,
          "error": null
        },
        {
          "field_type": "Currency",
          "valid": true,
          "error": null
        },
        {
          "field_type": "Picklist",
          "valid": true,
          "error": null
        },
        {
          "field_type": "Date",
          "valid": true,
          "error": null
        },
        {
          "field_type": "LongTextArea",
          "valid": t

In [ ]:
# ============================================================
# CELL 11 — Tool: check_structural_limits
#
# Counts what's already in the design — nothing invented,
# nothing estimated. Just counts fields and Master-Detail
# relationships per object and compares against real
# Salesforce metadata ceilings.
# ============================================================

MAX_FIELDS_PER_OBJECT             = 800
MAX_MASTER_DETAIL_PER_OBJECT       = 2


def check_structural_limits(obj: dict) -> dict:
    field_count = len(obj.get("fields", []))
    md_count = sum(
        1 for r in obj.get("relationships", [])
        if "master-detail" in r.lower() or "master detail" in r.lower()
    )

    errors = []
    if field_count > MAX_FIELDS_PER_OBJECT:
        errors.append(f"{field_count} fields exceeds max of {MAX_FIELDS_PER_OBJECT}")
    if md_count > MAX_MASTER_DETAIL_PER_OBJECT:
        errors.append(f"{md_count} Master-Detail relationships exceeds max of {MAX_MASTER_DETAIL_PER_OBJECT}")

    return {
        "api_name": obj.get("api_name"),
        "field_count": field_count,
        "master_detail_count": md_count,
        "passed": len(errors) == 0,
        "errors": errors
    }


print("✅ check_structural_limits() defined")

✅ check_structural_limits() defined


In [ ]:
# ============================================================
# CELL 12 — Test check_structural_limits() directly
# ============================================================

# A fine object — well under both limits
fine_object = {
    "api_name": "Loan_Application__c",
    "fields": ["A__c", "B__c", "C__c"],
    "relationships": ["Lookup to Account"]
}
print(check_structural_limits(fine_object))

# An object that breaks the Master-Detail limit on purpose
broken_object = {
    "api_name": "Test__c",
    "fields": ["A__c"],
    "relationships": ["Master-Detail to Account", "Master-Detail to Contact", "Master-Detail to User"]
}
print(check_structural_limits(broken_object))

{'api_name': 'Loan_Application__c', 'field_count': 3, 'master_detail_count': 0, 'passed': True, 'errors': []}
{'api_name': 'Test__c', 'field_count': 1, 'master_detail_count': 3, 'passed': False, 'errors': ['3 Master-Detail relationships exceeds max of 2']}


In [ ]:
# ============================================================
# CELL 13 — Node 3: validate
#
# Takes the dict design() already returned. Does NOT call
# the LLM. Just inspects what's already there — naming_valid,
# field_check_results — and adds structural limits on top.
# Produces one decision: passed True/False, plus a list of
# every specific error found, across every object.
# ============================================================

def validate(design_result: dict) -> dict:
    all_errors = []
    structural_results = []

    for obj in design_result.get("data_model", []):
        # naming, already checked inside design()
        if not obj.get("naming_valid", True):
            for err in obj.get("naming_errors", []):
                all_errors.append(f"{obj['api_name']}: {err}")

        # field types, already checked inside design()
        for field_result in obj.get("field_check_results", []):
            if not field_result.get("valid", True):
                all_errors.append(f"{obj['api_name']}: {field_result['error']}")

        # structural limits, checked here for the first time
        structural = check_structural_limits(obj)
        structural_results.append(structural)
        if not structural["passed"]:
            for err in structural["errors"]:
                all_errors.append(f"{obj['api_name']}: {err}")

    return {
        "passed": len(all_errors) == 0,
        "errors": all_errors,
        "structural_results": structural_results
    }


print("✅ validate() defined")

✅ validate() defined


In [ ]:
# ============================================================
# CELL 14 — Test validate() on the design result from Cell 10
# ============================================================

validation_result = validate(result)

print(json.dumps(validation_result, indent=2))

{
  "passed": true,
  "errors": [],
  "structural_results": [
    {
      "api_name": "Loan_Application__c",
      "field_count": 5,
      "master_detail_count": 0,
      "passed": true,
      "errors": []
    },
    {
      "api_name": "Co_Applicant__c",
      "field_count": 3,
      "master_detail_count": 0,
      "passed": true,
      "errors": []
    }
  ]
}


In [ ]:
# ============================================================
# CELL 15 — Node 4: output
#
# Takes the original requirement/current_process plus
# whatever design() and validate() already produced.
# Writes the problem/current_state/proposed_solution narrative
# with one small LLM call, then assembles the full final dict.
# Does not re-design anything, does not re-validate anything.
# ============================================================

OUTPUT_PROMPT = """You are a senior Salesforce Data Architect writing the
narrative sections of a design document. You are NOT designing anything new
here — only describing what was already decided.

REQUIREMENT: {requirement}
CURRENT PROCESS: {current_process}
PROPOSED DATA MODEL: {data_model_summary}
VALIDATION STATUS: {validation_status}

Respond in this exact JSON format only, no other text:
{{
  "problem": "1-2 sentence problem statement",
  "current_state": "1-2 sentences on what happens today",
  "proposed_solution": "1-2 sentences plainly describing the data model decision"
}}"""


def output(requirement: str, current_process: str,
           design_result: dict, validation_result: dict,
           clarifying_questions: list = None) -> dict:

    # If intake already flagged this as unclear, skip everything else
    if clarifying_questions:
        return {
            "problem": None,
            "current_state": None,
            "proposed_solution": None,
            "data_model": [],
            "validation_status": "needs_clarification",
            "alternative_considered": "",
            "confidence_score": 0.0,
            "clarifying_questions": clarifying_questions
        }

    client = OpenAI()

    data_model_summary = ", ".join(
        f"{obj['object_name']} ({len(obj.get('fields', []))} fields)"
        for obj in design_result.get("data_model", [])
    )

    validation_status = "passed" if validation_result["passed"] else "failed"

    prompt = OUTPUT_PROMPT.format(
        requirement=requirement,
        current_process=current_process,
        data_model_summary=data_model_summary,
        validation_status=validation_status
    )

    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=300,
        temperature=0.2
    )

    raw = resp.choices[0].message.content.strip()
    raw = re.sub(r"^```[a-z]*\n?|\n?```$", "", raw)
    narrative = json.loads(raw)

    # confidence is simple and honest: 1.0 if validation passed clean,
    # lower if there were errors design had to work through
    confidence = 1.0 if validation_result["passed"] else 0.5

    return {
        "problem": narrative["problem"],
        "current_state": narrative["current_state"],
        "proposed_solution": narrative["proposed_solution"],
        "data_model": design_result.get("data_model", []),
        "validation_status": validation_status,
        "alternative_considered": design_result.get("alternative_considered", ""),
        "confidence_score": confidence,
        "clarifying_questions": []
    }


print("✅ output() defined")

✅ output() defined


In [ ]:
# ============================================================
# CELL 16 — Test output() on everything produced so far
# ============================================================

final_result = output(
    requirement="Track loan applications from submission through underwriting approval.",
    current_process="Currently tracked in spreadsheets.",
    design_result=result,
    validation_result=validation_result
)

print(json.dumps(final_result, indent=2))

{
  "problem": "The current process for tracking loan applications is inefficient and prone to errors due to reliance on spreadsheets. This lack of a centralized system hinders visibility and accountability throughout the loan application process.",
  "current_state": "Currently, loan applications are tracked using spreadsheets, which can lead to data inconsistencies and difficulties in collaboration. This method does not provide a streamlined approach for managing the application lifecycle from submission to underwriting approval.",
  "proposed_solution": "The proposed data model includes a Loan Application object with five fields to capture essential application details and a Co-Applicant object with three fields to manage information related to co-applicants. This structure aims to enhance data integrity and streamline the tracking process.",
  "data_model": [
    {
      "object_name": "Loan Application",
      "api_name": "Loan_Application__c",
      "fields": [
        "Applicati

In [ ]:
# ============================================================
# CELL 17 — The full run, wiring all four nodes together
#
# No LangGraph yet — just plain Python control flow, so you
# can see exactly what decides what happens next, with nothing
# hidden behind a framework.
# ============================================================

def run_agent(requirement: str, process_flow: str, edge_cases: str,
              current_process: str, override_note: str = "",
              max_retries: int = 3) -> dict:

    # Step 1 — intake
    intake_result = intake(
        requirement=requirement,
        process_flow=process_flow,
        edge_cases=edge_cases,
        current_process=current_process
    )

    if not intake_result["is_clear"]:
        return output(
            requirement=requirement,
            current_process=current_process,
            design_result={},
            validation_result={"passed": False, "errors": []},
            clarifying_questions=intake_result["clarifying_questions"]
        )

    # Step 2 — design, with retry loop driven by validate's feedback
    retry_note = override_note
    for attempt in range(max_retries):
        design_result = design(
            requirement=requirement,
            process_flow=process_flow,
            edge_cases=edge_cases,
            current_process=current_process,
            override_note=retry_note
        )

        validation_result = validate(design_result)

        if validation_result["passed"]:
            break

        # Failed — feed the specific errors back in as the next
        # override_note, so design() tries again with that context
        error_summary = "; ".join(validation_result["errors"])
        retry_note = f"Previous attempt had these errors, fix them: {error_summary}"
        print(f"  Attempt {attempt + 1} failed, retrying: {error_summary}")

    # Step 3 — output, regardless of whether validation ultimately passed
    return output(
        requirement=requirement,
        current_process=current_process,
        design_result=design_result,
        validation_result=validation_result
    )


print("✅ run_agent() defined")

✅ run_agent() defined


In [ ]:
# ============================================================
# CELL 18 — Run the whole thing end to end, one call
# ============================================================

final = run_agent(
    requirement="Track loan applications from submission through underwriting approval.",
    process_flow="Applicant submits, credit check runs (need object), financial documents recieved, underwriter reviews credit check and uploaded documents, decision made, applicant notified.",
    edge_cases="Some applications need co-applicants.",
    current_process="Currently tracked in spreadsheets."
)

print(json.dumps(final, indent=2))

{
  "problem": "The current process for tracking loan applications is inefficient and prone to errors due to reliance on spreadsheets. This lack of a centralized system hinders visibility and accountability throughout the application process.",
  "current_state": "Currently, loan applications are tracked manually in spreadsheets, which can lead to inconsistencies and difficulties in managing the workflow. This method does not provide real-time updates or a comprehensive view of the application status.",
  "proposed_solution": "The proposed data model includes three main entities: Loan Application with six fields to capture essential details, Credit Check with three fields to track credit-related information, and Co-Applicant with two fields for additional applicant details. This structured approach will streamline the tracking process from submission through underwriting approval.",
  "data_model": [
    {
      "object_name": "Loan Application",
      "api_name": "Loan_Application__c"

In [ ]:
# ============================================================
# CELL 19 — Run the whole thing end to end, one call with a vaugue ask.
# ============================================================

final = run_agent(
    requirement="Track our stuff",
    process_flow="Somethings happen then its approved",
    edge_cases="We might have some",
    current_process="Currently tracked in spreadsheets."
)

print(json.dumps(final, indent=2))

{
  "problem": null,
  "current_state": null,
  "proposed_solution": null,
  "data_model": [],
  "validation_status": "needs_clarification",
  "alternative_considered": "",
  "confidence_score": 0.0,
  "clarifying_questions": [
    "What specific items or data need to be tracked?",
    "What are the steps involved in the approval process?",
    "What are the edge cases that need to be considered?",
    "Which systems or users will interact with this tracking process?",
    "What specific information is currently being tracked in the spreadsheets?"
  ]
}


In [ ]:
# ============================================================
# CELL 20 — Focus validation to fail
# ============================================================

final = run_agent(
    requirement="Track loan applications from submission through underwriting approval.",
    process_flow="Applicant submits, credit check runs (need object), financial documents recieved, underwriter reviews credit check and uploaded documents, decision made, applicant notified.",
    edge_cases="Some applications need co-applicants. And the credit object must end with __e, it is a hard requirement for a downstream integration",
    current_process="Currently tracked in spreadsheets."
)

print(json.dumps(final, indent=2))

  Attempt 1 failed, retrying: Credit_Check__e: Custom object names must end with __c
{
  "problem": "The current process of tracking loan applications relies on spreadsheets, which can lead to inefficiencies and errors in data management. There is a need for a more structured and reliable system to monitor the progress of loan applications from submission to underwriting approval.",
  "current_state": "Currently, loan applications are tracked manually in spreadsheets, making it difficult to maintain accurate records and streamline the workflow. This method lacks real-time updates and collaboration features, resulting in potential delays and miscommunication.",
  "proposed_solution": "The proposed data model includes three main entities: Loan Application with four fields, Credit Check with three fields, and Co-Applicant with two fields. This structured approach will facilitate better tracking and management of loan applications throughout the underwriting process.",
  "data_model": [
  

In [ ]:
# ============================================================
# CELL 21 — Extraction layer (UI-only, not part of run_agent)
#
# Takes one freeform paragraph from a user
# and produces the four clean fields run_agent() actually needs.
# This is NOT a node in the agent. It exists only so the public
# UI can have a single input box. The agent itself stays exactly
# as built — four clean strings in, same as before.
# ============================================================

EXTRACTION_PROMPT = """A user wrote a rough description of a business need.
Extract four things from it. If something genuinely isn't mentioned, write
"Not specified" rather than inventing detail.

USER INPUT: {raw_text}

Respond in this exact JSON format only, no other text:
{{
  "requirement": "the core business need, restated clearly",
  "process_flow": "the sequence of steps described, if any",
  "edge_cases": "exceptions or unusual scenarios mentioned, if any",
  "current_process": "how it's done today, if mentioned"
}}"""


def extract_fields(raw_text: str) -> dict:
    client = OpenAI()

    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": EXTRACTION_PROMPT.format(raw_text=raw_text)}],
        max_tokens=400,
        temperature=0.1
    )

    raw = resp.choices[0].message.content.strip()
    raw = re.sub(r"^```[a-z]*\n?|\n?```$", "", raw)
    return json.loads(raw)


print("✅ extract_fields() defined")

✅ extract_fields() defined


In [ ]:
# ============================================================
# CELL 22 — Test extraction on a messy, single-paragraph input
# ============================================================

messy_input = """We're a hospital network and right now when a primary care
doctor wants to send a patient to a specialist, it's all done over fax and
phone calls, nobody can see the status. We need this tracked properly,
from when the doctor refers someone all the way until the patient actually
sees the specialist. Sometimes the specialist's office is full and the
referral has to get redirected to a different doctor."""

extracted = extract_fields(messy_input)
print(json.dumps(extracted, indent=2))

{
  "requirement": "To track patient referrals from primary care doctors to specialists properly.",
  "process_flow": "From when the doctor refers someone to when the patient sees the specialist.",
  "edge_cases": "Referrals may need to be redirected to a different doctor if the specialist's office is full.",
  "current_process": "Referrals are currently done over fax and phone calls, with no visibility on status."
}
